<a href="https://colab.research.google.com/github/HariniAnandkumar/neuromorphic-ai-research/blob/main/02_snn_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [ ]:
!pip install snntorch -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import snntorch as snn
from snntorch import spikegen
import time
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 9.8 MB/s eta 0:00:00
Device: cuda


In [ ]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)

testset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.60MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 141kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.02MB/s]


In [ ]:
num_steps = 10
beta = 0.95

class SNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3)
        self.lif1 = snn.Leaky(beta=beta)

        self.conv2 = nn.Conv2d(32, 64, 3)
        self.lif2 = snn.Leaky(beta=beta)

        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(9216, 128)
        self.lif3 = snn.Leaky(beta=beta)

        self.fc2 = nn.Linear(128, 10)
        self.lif4 = snn.Leaky(beta=beta)

    def forward(self, x):   # <-- MUST be indented inside class

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()

        spk1_rec = []
        spk2_rec = []
        spk3_rec = []
        spk4_rec = []

        for step in range(num_steps):

            cur1 = self.conv1(x)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.conv2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            pooled = self.pool(spk2)
            flat = self.flatten(pooled)

            cur3 = self.fc1(flat)
            spk3, mem3 = self.lif3(cur3, mem3)

            cur4 = self.fc2(spk3)
            spk4, mem4 = self.lif4(cur4, mem4)

            spk1_rec.append(spk1)
            spk2_rec.append(spk2)
            spk3_rec.append(spk3)
            spk4_rec.append(spk4)

        return (
            torch.stack(spk1_rec),
            torch.stack(spk2_rec),
            torch.stack(spk3_rec),
            torch.stack(spk4_rec)
        )
        model = SNN().to(device)

In [ ]:
model = SNN().to(device)
print("Model created successfully")

Model created successfully


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 10
start_time = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        spk1, spk2, spk3, spk4 = model(images)

        # Sum over time
        output_sum = spk4.sum(dim=0)

        loss = criterion(output_sum, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(trainloader):.4f}")

train_time = time.time() - start_time
print("Training Time (sec):", train_time)

Epoch 1/10 | Loss: 0.1399
Epoch 2/10 | Loss: 0.0403
Epoch 3/10 | Loss: 0.0236
Epoch 4/10 | Loss: 0.0181
Epoch 5/10 | Loss: 0.0129
Epoch 6/10 | Loss: 0.0086
Epoch 7/10 | Loss: 0.0083
Epoch 8/10 | Loss: 0.0085
Epoch 9/10 | Loss: 0.0088
Epoch 10/10 | Loss: 0.0074
Training Time (sec): 565.0752935409546


In [ ]:
model.eval()
correct = 0
total = 0
total_spikes = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        spk1, spk2, spk3, spk4 = model(images)
        output_sum = spk4.sum(dim=0)

        _, predicted = torch.max(output_sum, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        total_spikes += (
            spk1.sum() +
            spk2.sum() +
            spk3.sum() +
            spk4.sum()
        ).item()

test_acc = 100 * correct / total
print("Test Accuracy:", test_acc)
print("Total Spikes:", total_spikes)

Test Accuracy: 98.91
Total Spikes: 189941040.0


In [ ]:
metrics = {
    "model": "SNN",
    "accuracy": test_acc,
    "train_time_sec": train_time,
    "params": sum(p.numel() for p in model.parameters()),
    "flops": None,
    "spike_count": total_spikes
}

df = pd.DataFrame([metrics])
df.to_csv("snn_mnist_results.csv", index=False)

print("Saved snn_mnist_results.csv")

Saved snn_mnist_results.csv
